# MobileNetV3-Small — Waste Classification (Checkpoint 1)

Paper: Yong, L., Ma, L., Sun, D., & Du, L. (2023). *Application of MobileNetV2 to waste classification.*
PLOS ONE 18(3): e0282336. <https://doi.org/10.1371/journal.pone.0282336>

Notebook ini menjalankan **MobileNetV3-Small** pada dataset, split, dan protokol training yang sama
dengan eksperimen MobileNetV2 dan CNN. Semua artefak ditulis ke `outputs/` di root repository.

## Daftar isi

1. Setup dan konfigurasi
2. Dataset dan split
3. Transform dan loader
4. Model MobileNetV3-Small
5. Training loop
6. Training MobileNetV3-Small
7. Kurva training
8. Evaluasi test
9. Confusion matrix dan inference
10. Analisis error
11. Perbandingan dengan baseline
12. Simpan ringkasan

## 1. Setup dan konfigurasi

Hyperparameter training mengikuti `train_245_class.py` (SGD 0,001, momentum 0,9, StepLR 7/0,1,
50 epoch, batch 42) agar hasil MobileNetV3-Small sebanding dengan MobileNetV2 dan CNN.

In [ ]:
# ---------------------------------------------------------------------------
# Auto-install dependencies. Packages that are already installed are untouched.
# ---------------------------------------------------------------------------
from __future__ import annotations

import importlib.util
import subprocess
import sys


def module_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def pip_install(*packages: str, index_url: str | None = None) -> bool:
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if index_url:
        command += ["--index-url", index_url]
    command += list(packages)
    print("    $", " ".join(command))
    try:
        return subprocess.run(command, check=False).returncode == 0
    except Exception as exc:
        print(f"[warn] pip install failed for {' '.join(packages)}: {exc}")
        return False


def ensure(packages: dict[str, str]) -> list[str]:
    missing = [pkg for pkg, mod in packages.items() if not module_available(mod)]
    if not missing:
        return []
    print(f"[setup] installing: {', '.join(missing)}")
    for package in missing:
        pip_install(package)
    still_missing = [pkg for pkg in missing if not module_available(packages[pkg])]
    if still_missing:
        print("[warn] could not install: " + ", ".join(still_missing))
    return still_missing


ensure({
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scikit-learn": "sklearn",
    "Pillow": "PIL",
    "tqdm": "tqdm",
    "ipywidgets": "ipywidgets",
})

if not module_available("torch") or not module_available("torchvision"):
    print("[setup] installing torch and torchvision")
    installed = pip_install(
        "torch", "torchvision", index_url="https://download.pytorch.org/whl/cpu"
    )
    if not installed and (not module_available("torch") or not module_available("torchvision")):
        print("[setup] retrying with the default PyPI index")
        pip_install("torch", "torchvision")

print("[setup] dependency check done")

In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
try:
    from ipywidgets import IProgress  # noqa: F401
    from tqdm.auto import tqdm
except Exception:
    from tqdm import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score,
)

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "axes.unicode_minus": False,
})

SEED = 42


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything()


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "outputs").is_dir() and (candidate / "Datasets").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = REPO_ROOT / "outputs"
SPLIT_DIR = OUTPUT_DIR / "splits"
MODEL_DIR = OUTPUT_DIR / "models"
RESULT_DIR = OUTPUT_DIR / "results"
FIGURE_DIR = OUTPUT_DIR / "figures"
for _d in (OUTPUT_DIR, SPLIT_DIR, MODEL_DIR, RESULT_DIR, FIGURE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path(
    os.environ.get("TRASH_DATA_DIR", REPO_ROOT / "Datasets" / "images" / "images")
).resolve()

# Paper defaults from train_245_class.py, kept identical for a fair comparison.
IMG_SIZE = 224
BATCH_SIZE = 42
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
MOMENTUM = 0.9
STEP_SIZE = 7
GAMMA = 0.1
NUM_WORKERS = 0
PRETRAINED = os.environ.get("TRASH_REPRO_PRETRAINED", "1") == "1"
TEST_BATCH_SIZE = 128

QUICK_TEST = os.environ.get("TRASH_REPRO_QUICK", "0") == "1"
TRAIN_PER_CLASS = VAL_PER_CLASS = TEST_PER_CLASS = None
if QUICK_TEST:
    NUM_EPOCHS = 1
    BATCH_SIZE = 16
    PRETRAINED = False
    TRAIN_PER_CLASS, VAL_PER_CLASS, TEST_PER_CLASS = 20, 5, 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python           : {sys.version.split()[0]}")
print(f"PyTorch          : {torch.__version__}")
print(f"Device           : {DEVICE}")
print(f"Repo root        : {REPO_ROOT}")
print(f"Data root        : {DATA_ROOT} (exists={DATA_ROOT.exists()})")
print(f"Output folder    : {OUTPUT_DIR}")
print(f"Quick test       : {QUICK_TEST}")
print(f"Pretrained weights: {PRETRAINED}")

## 2. Dataset dan split

Split 8:1:1 yang dipakai MobileNetV2 dan CNN dibaca ulang dari `outputs/splits/all.csv`
(stratified per kelas dan source group, seed 42). Dataset dan splitnya identik, jadi dedupe MD5
dan split tidak diulang di sini.

In [ ]:
# ---------------------------------------------------------------------------
# 30 fine classes and the 3-class grouping used by the earlier experiments.
# ---------------------------------------------------------------------------
CLASS_NAMES = [
    "aerosol_cans",
    "aluminum_food_cans",
    "aluminum_soda_cans",
    "cardboard_boxes",
    "cardboard_packaging",
    "clothing",
    "coffee_grounds",
    "disposable_plastic_cutlery",
    "eggshells",
    "food_waste",
    "glass_beverage_bottles",
    "glass_cosmetic_containers",
    "glass_food_jars",
    "magazines",
    "newspaper",
    "office_paper",
    "paper_cups",
    "plastic_cup_lids",
    "plastic_detergent_bottles",
    "plastic_food_containers",
    "plastic_shopping_bags",
    "plastic_soda_bottles",
    "plastic_straws",
    "plastic_trash_bags",
    "plastic_water_bottles",
    "shoes",
    "steel_food_cans",
    "styrofoam_cups",
    "styrofoam_food_containers",
    "tea_bags",
]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

MAJOR_NAMES = ["Kitchen waste", "Recyclable waste", "Other waste"]
MAJOR_TO_IDX = {name: i for i, name in enumerate(MAJOR_NAMES)}

CLASS_TO_MAJOR = {
    "coffee_grounds": "Kitchen waste",
    "eggshells": "Kitchen waste",
    "food_waste": "Kitchen waste",
    "tea_bags": "Kitchen waste",
    "aerosol_cans": "Recyclable waste",
    "aluminum_food_cans": "Recyclable waste",
    "aluminum_soda_cans": "Recyclable waste",
    "cardboard_boxes": "Recyclable waste",
    "cardboard_packaging": "Recyclable waste",
    "clothing": "Recyclable waste",
    "glass_beverage_bottles": "Recyclable waste",
    "glass_cosmetic_containers": "Recyclable waste",
    "glass_food_jars": "Recyclable waste",
    "magazines": "Recyclable waste",
    "newspaper": "Recyclable waste",
    "office_paper": "Recyclable waste",
    "paper_cups": "Recyclable waste",
    "plastic_detergent_bottles": "Recyclable waste",
    "plastic_food_containers": "Recyclable waste",
    "plastic_shopping_bags": "Recyclable waste",
    "plastic_soda_bottles": "Recyclable waste",
    "plastic_water_bottles": "Recyclable waste",
    "shoes": "Recyclable waste",
    "steel_food_cans": "Recyclable waste",
    "disposable_plastic_cutlery": "Other waste",
    "plastic_cup_lids": "Other waste",
    "plastic_straws": "Other waste",
    "plastic_trash_bags": "Other waste",
    "styrofoam_cups": "Other waste",
    "styrofoam_food_containers": "Other waste",
}

missing_mapping = sorted(set(CLASS_NAMES) - set(CLASS_TO_MAJOR))
assert not missing_mapping, f"Missing major-class mapping for: {missing_mapping}"

print(f"Classes: {NUM_CLASSES}")
print(pd.Series(CLASS_TO_MAJOR).value_counts().to_string())

In [ ]:
assert DATA_ROOT.exists(), f"Dataset folder not found: {DATA_ROOT}. Set TRASH_DATA_DIR."
missing_dirs = [name for name in CLASS_NAMES if not (DATA_ROOT / name).is_dir()]
assert not missing_dirs, f"Missing class folders under {DATA_ROOT}: {missing_dirs}"

SPLIT_MANIFEST = SPLIT_DIR / "all.csv"
if not SPLIT_MANIFEST.exists():
    raise FileNotFoundError(
        f"{SPLIT_MANIFEST} not found. Run the MobileNetV2 notebook once to write the manifest."
    )

df = pd.read_csv(SPLIT_MANIFEST)
df = df[["rel_path", "class_name", "source_group", "md5", "split", "label"]]
df["label"] = df["class_name"].map(CLASS_TO_IDX).astype(int)

if QUICK_TEST:
    limits = {"train": TRAIN_PER_CLASS, "val": VAL_PER_CLASS, "test": TEST_PER_CLASS}
    parts = []
    for (split_name, _class_name), group in df.groupby(["split", "class_name"], sort=True):
        n_total = limits[split_name]
        n_groups = max(1, group["source_group"].nunique())
        n_per_group = max(1, math.ceil(n_total / n_groups))
        for _source_group, subgroup in group.groupby("source_group", sort=True):
            parts.append(subgroup.head(n_per_group))
    df = pd.concat(parts, ignore_index=True)
    trimmed = []
    for (split_name, _class_name), group in df.groupby(["split", "class_name"], sort=True):
        trimmed.append(group.head(limits[split_name]))
    df = pd.concat(trimmed, ignore_index=True)

df = df.sort_values(["split", "class_name", "rel_path"]).reset_index(drop=True)

print("Split sizes:")
print(df["split"].value_counts().reindex(["train", "val", "test"]).to_string())
assert set(df["split"]) == {"train", "val", "test"}
assert df.groupby("split")["class_name"].nunique().eq(NUM_CLASSES).all()
assert df.groupby("md5")["split"].nunique().eq(1).all(), "Duplicate image crossed a split."

## 3. Transform dan loader

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class TrashDataset(Dataset):
    '''Manifest-backed image dataset for the Kaggle waste folder layout.'''

    def __init__(self, frame: pd.DataFrame, root: Path, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.root = Path(root)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        with Image.open(self.root / row["rel_path"]) as img:
            image = img.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["label"])


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(frame: pd.DataFrame, training: bool, batch_size: int) -> DataLoader:
    dataset = TrashDataset(
        frame, DATA_ROOT, transform=train_transform if training else eval_transform
    )
    # BatchNorm needs more than one value per channel.
    drop_last = (
        training and len(dataset) >= batch_size and len(dataset) % batch_size == 1
    )
    generator = torch.Generator()
    generator.manual_seed(SEED)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=training,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=drop_last,
        persistent_workers=NUM_WORKERS > 0,
        worker_init_fn=seed_worker,
        generator=generator,
    )


train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

train_loader = make_loader(train_df, training=True, batch_size=BATCH_SIZE)
val_loader = make_loader(val_df, training=False, batch_size=BATCH_SIZE)
test_loader = make_loader(test_df, training=False, batch_size=TEST_BATCH_SIZE)

print(f"Train: {len(train_df):,} images, {len(train_loader)} batches")
print(f"Val  : {len(val_df):,} images, {len(val_loader)} batches")
print(f"Test : {len(test_df):,} images, {len(test_loader)} batches")

images, labels = next(iter(train_loader))
print("Batch shape:", tuple(images.shape))
print("Label range:", int(labels.min()), "..", int(labels.max()))

## 4. Model MobileNetV3-Small

`torchvision.models.mobilenet_v3_small` dimuat dengan bobot ImageNet. Layer terakhir classifier
diganti menjadi linear 30 kelas, seluruh parameter di-fine-tune.

In [ ]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_mobilenet_v3_small(num_classes: int, pretrained: bool = True) -> nn.Module:
    weights = "IMAGENET1K_V1" if pretrained else None
    model = models.mobilenet_v3_small(weights=weights)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, num_classes)
    return model


model = build_mobilenet_v3_small(NUM_CLASSES, pretrained=PRETRAINED).to(DEVICE)
print(model.classifier)
print(f"Trainable parameters: {count_parameters(model):,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)
scheduler = lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)

print(f"Optimizer: SGD(lr={LEARNING_RATE}, momentum={MOMENTUM})")
print(f"Scheduler: StepLR(step_size={STEP_SIZE}, gamma={GAMMA})")

## 5. Training loop

Epoch terbaik dipilih berdasarkan **validation accuracy** tertinggi, sama seperti
`train_245_class.py`. Checkpoint berisi `state_dict` plus metadata.

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    epochs,
    device,
    checkpoint_path,
    history_path,
    model_name,
    arch,
    extra_config=None,
    full_model_path=None,
):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0
    best_epoch = 0
    history = []
    dataloaders = {"train": train_loader, "val": val_loader}
    dataset_sizes = {phase: len(loader.dataset) for phase, loader in dataloaders.items()}

    for epoch in range(1, epochs + 1):
        print(f"Epoch {epoch}/{epochs}")
        print("-" * 10)
        epoch_start = time.time()
        epoch_losses = {}
        epoch_accs = {}

        for phase in ["train", "val"]:
            model.train() if phase == "train" else model.eval()

            running_loss = 0.0
            running_corrects = 0
            loader = dataloaders[phase]

            progress = tqdm(
                total=dataset_sizes[phase],
                desc=f"{model_name} {phase} {epoch}/{epochs}",
                leave=False,
            )
            for inputs, labels in loader:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                progress.update(inputs.size(0))
            progress.close()

            if phase == "train" and scheduler is not None:
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            epoch_losses[phase] = float(epoch_loss)
            epoch_accs[phase] = float(epoch_acc)
            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            if phase == "val" and epoch_acc > best_val_acc:
                best_val_acc = float(epoch_acc)
                best_epoch = epoch
                best_model_wts = copy.deepcopy(model.state_dict())
                checkpoint = {
                    "arch": arch,
                    "model_name": model_name,
                    "state_dict": model.state_dict(),
                    "class_names": CLASS_NAMES,
                    "major_names": MAJOR_NAMES,
                    "class_to_major": CLASS_TO_MAJOR,
                    "img_size": IMG_SIZE,
                    "mean": IMAGENET_MEAN,
                    "std": IMAGENET_STD,
                    "epoch": epoch,
                    "val_acc": float(epoch_acc),
                    "val_loss": float(epoch_loss),
                    "config": {
                        "seed": SEED,
                        "batch_size": BATCH_SIZE,
                        "epochs": epochs,
                        "pretrained": PRETRAINED,
                        **(extra_config or {}),
                    },
                }
                torch.save(checkpoint, checkpoint_path)

        current_lr = optimizer.param_groups[0]["lr"]
        record = {
            "epoch": epoch,
            "train_loss": epoch_losses["train"],
            "train_acc": epoch_accs["train"],
            "val_loss": epoch_losses["val"],
            "val_acc": epoch_accs["val"],
            "lr": float(current_lr),
            "time_sec": float(time.time() - epoch_start),
        }
        history.append(record)
        pd.DataFrame(history).to_csv(history_path, index=False)

        print(
            f"[{model_name}] epoch {epoch:02d}/{epochs} | "
            f"train loss {record['train_loss']:.4f} acc {record['train_acc']:.4f} | "
            f"val loss {record['val_loss']:.4f} acc {record['val_acc']:.4f} | "
            f"lr {current_lr:.2e} | {record['time_sec']:.1f}s"
        )
        print()

    time_elapsed = time.time() - since
    print(f"Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Best val Acc: {best_val_acc:.4f} at epoch {best_epoch}")
    model.load_state_dict(best_model_wts)
    if full_model_path is not None:
        torch.save(model, full_model_path)
        print(f"Full model: {full_model_path}")
    print(f"Checkpoint: {checkpoint_path}")
    return history, best_val_acc


def load_checkpoint(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)
    except Exception:
        return torch.load(path, map_location=device, weights_only=False)

## 6. Training MobileNetV3-Small

In [ ]:
CHECKPOINT = MODEL_DIR / "mobilenetv3small_trash30_best.pt"
FULL_MODEL = MODEL_DIR / "mobilenetv3small_trash30_full.pt"
HISTORY_CSV = RESULT_DIR / "mobilenetv3small_history.csv"

history, best_val_acc = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=NUM_EPOCHS,
    device=DEVICE,
    checkpoint_path=CHECKPOINT,
    full_model_path=FULL_MODEL,
    history_path=HISTORY_CSV,
    model_name="MobileNetV3-Small",
    arch="mobilenet_v3_small",
    extra_config={
        "learning_rate": LEARNING_RATE,
        "momentum": MOMENTUM,
        "step_size": STEP_SIZE,
        "gamma": GAMMA,
    },
)

print(f"Best validation accuracy: {best_val_acc:.4f}")

## 7. Kurva training

In [ ]:
def plot_training_curves(history, title: str, output_path: Path):
    frame = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].plot(frame["epoch"], frame["train_loss"], marker="o", ms=3, label="Train")
    axes[0].plot(frame["epoch"], frame["val_loss"], marker="o", ms=3, label="Validation")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("cross-entropy loss")
    axes[0].set_title("Loss")
    axes[0].legend()

    axes[1].plot(frame["epoch"], frame["train_acc"], marker="o", ms=3, label="Train")
    axes[1].plot(frame["epoch"], frame["val_acc"], marker="o", ms=3, label="Validation")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("accuracy")
    axes[1].set_title("Accuracy")
    axes[1].legend()

    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)


plot_training_curves(
    history,
    "MobileNetV3-Small — training and validation",
    FIGURE_DIR / "mobilenetv3small_curves.png",
)
display(pd.DataFrame(history).tail())

## 8. Evaluasi test

In [ ]:
def jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, Path):
        return str(obj)
    return obj


def aggregate_major_probs(probs: np.ndarray, class_names):
    major_probs = np.zeros((probs.shape[0], len(MAJOR_NAMES)), dtype=np.float64)
    for class_idx, class_name in enumerate(class_names):
        major_probs[:, MAJOR_TO_IDX[CLASS_TO_MAJOR[class_name]]] += probs[:, class_idx]
    return major_probs


@torch.no_grad()
def predict_loader(model, loader, criterion, device):
    model.eval()
    all_probs, all_labels = [], []
    total_loss = 0.0
    total_samples = 0

    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        total_loss += loss.item() * labels.size(0)
        total_samples += labels.size(0)

    if total_samples == 0:
        raise RuntimeError("Empty evaluation loader.")
    probs = np.concatenate(all_probs, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    return probs, labels, total_loss / total_samples


def plot_confusion_matrix(cm, labels, title, output_path, normalize=True):
    cm = np.asarray(cm, dtype=np.float64)
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        plot_cm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)
        fmt = ".2f"
    else:
        plot_cm = cm
        fmt = "d"

    annot = len(labels) <= 12
    fig, ax = plt.subplots(figsize=(max(7, len(labels) * 0.48), max(6, len(labels) * 0.42)))
    sns.heatmap(
        plot_cm,
        annot=annot,
        fmt=fmt,
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
        cbar=True,
        linewidths=0.2 if annot else 0,
        annot_kws={"fontsize": 9} if annot else None,
    )
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    ax.tick_params(axis="y", rotation=0, labelsize=7)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def evaluate_and_save(model, loader, criterion, device, frame, model_key):
    probs, labels, loss = predict_loader(model, loader, criterion, device)
    preds = probs.argmax(axis=1)

    report_dict = classification_report(
        labels,
        preds,
        labels=np.arange(NUM_CLASSES),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(labels, preds, labels=np.arange(NUM_CLASSES))
    per_class_acc = np.divide(
        cm.diagonal(),
        cm.sum(axis=1),
        out=np.zeros(NUM_CLASSES, dtype=np.float64),
        where=cm.sum(axis=1) != 0,
    )

    major_probs = aggregate_major_probs(probs, CLASS_NAMES)
    major_true = frame["class_name"].map(lambda c: MAJOR_TO_IDX[CLASS_TO_MAJOR[c]]).to_numpy()
    major_pred = np.array([
        MAJOR_TO_IDX[CLASS_TO_MAJOR[CLASS_NAMES[int(index)]]] for index in preds
    ])
    major_prob_pred = major_probs.argmax(axis=1)
    major_report = classification_report(
        major_true,
        major_pred,
        labels=np.arange(len(MAJOR_NAMES)),
        target_names=MAJOR_NAMES,
        output_dict=True,
        zero_division=0,
    )
    major_cm = confusion_matrix(major_true, major_pred, labels=np.arange(len(MAJOR_NAMES)))

    prediction_frame = frame[["rel_path", "class_name", "source_group"]].copy()
    prediction_frame["true_label"] = labels
    prediction_frame["pred_label"] = preds
    prediction_frame["pred_class"] = [CLASS_NAMES[i] for i in preds]
    prediction_frame["confidence"] = probs.max(axis=1)
    prediction_frame["correct"] = (labels == preds).astype(int)
    for class_idx, class_name in enumerate(CLASS_NAMES):
        prediction_frame[f"prob_{class_name}"] = probs[:, class_idx]
    prediction_frame.to_csv(RESULT_DIR / f"{model_key}_predictions.csv", index=False)

    metrics = {
        "model": model_key,
        "split": "test",
        "num_test": int(len(labels)),
        "accuracy": float(accuracy_score(labels, preds)),
        "loss": float(loss),
        "top3_accuracy": float(
            top_k_accuracy_score(
                labels,
                probs,
                k=min(3, NUM_CLASSES),
                labels=np.arange(NUM_CLASSES),
            )
        ),
        "macro_f1": float(report_dict["macro avg"]["f1-score"]),
        "weighted_f1": float(report_dict["weighted avg"]["f1-score"]),
        "per_class_accuracy": {
            name: float(per_class_acc[i]) for i, name in enumerate(CLASS_NAMES)
        },
        "group_accuracy": {
            str(group): float(sub["correct"].mean())
            for group, sub in prediction_frame.groupby("source_group")
        },
        "classification_report": report_dict,
        "confusion_matrix": cm.tolist(),
        "major_accuracy": float(accuracy_score(major_true, major_pred)),
        "major_prob_accuracy": float(accuracy_score(major_true, major_prob_pred)),
        "major_report": major_report,
        "major_confusion_matrix": major_cm.tolist(),
        "major_class_names": MAJOR_NAMES,
        "class_names": CLASS_NAMES,
    }
    with open(RESULT_DIR / f"{model_key}_metrics.json", "w") as handle:
        json.dump(jsonable(metrics), handle, indent=2)

    pd.DataFrame(report_dict).T.to_csv(RESULT_DIR / f"{model_key}_classification_report.csv")
    pd.DataFrame(major_report).T.to_csv(RESULT_DIR / f"{model_key}_major_report.csv")
    pd.DataFrame(
        per_class_acc, index=CLASS_NAMES, columns=["accuracy"]
    ).to_csv(RESULT_DIR / f"{model_key}_per_class_accuracy.csv")

    print(f"\n{model_key} — test results")
    print(f"  loss              : {metrics['loss']:.4f}")
    print(f"  30-class accuracy : {metrics['accuracy'] * 100:.2f}%")
    print(f"  macro F1 (30)     : {metrics['macro_f1']:.4f}")
    print(f"  top-3 accuracy    : {metrics['top3_accuracy'] * 100:.2f}%")
    print(f"  3-class accuracy  : {metrics['major_accuracy'] * 100:.2f}% (top-1 prefix)")
    print(f"  3-class prob agg  : {metrics['major_prob_accuracy'] * 100:.2f}%")
    print("  accuracy by source group:")
    for group, value in metrics["group_accuracy"].items():
        print(f"    {group:10s}: {value * 100:.2f}%")

    return metrics, probs, labels, preds, major_true, major_pred

In [ ]:
best_model = build_mobilenet_v3_small(NUM_CLASSES, pretrained=False).to(DEVICE)
ckpt = load_checkpoint(CHECKPOINT, DEVICE)
best_model.load_state_dict(ckpt["state_dict"])
best_model.eval()
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (val acc {ckpt['val_acc']:.4f})")

(
    metrics,
    test_probs,
    test_labels,
    test_preds,
    major_true,
    major_pred,
) = evaluate_and_save(
    best_model,
    test_loader,
    criterion,
    DEVICE,
    test_df,
    model_key="mobilenetv3small",
)

display(pd.DataFrame(metrics["major_report"]).T.round(3))
display(pd.DataFrame(metrics["classification_report"]).T)

In [ ]:
insights = pd.DataFrame({
    "Metric": [
        "top-3 accuracy",
        "3-class accuracy (top-1 prefix)",
        "3-class accuracy (probability aggregation)",
        "macro F1",
    ],
    "Value": [
        metrics["top3_accuracy"],
        metrics["major_accuracy"],
        metrics["major_prob_accuracy"],
        metrics["macro_f1"],
    ],
})
display(insights.round(4))

## 9. Confusion matrix dan inference

In [ ]:
plot_confusion_matrix(
    np.array(metrics["confusion_matrix"]),
    CLASS_NAMES,
    "MobileNetV3-Small — 30-class confusion matrix (row-normalized)",
    FIGURE_DIR / "mobilenetv3small_confusion_30.png",
    normalize=True,
)

plot_confusion_matrix(
    np.array(metrics["major_confusion_matrix"]),
    MAJOR_NAMES,
    "MobileNetV3-Small — 3 major class confusion matrix",
    FIGURE_DIR / "mobilenetv3small_confusion_major.png",
    normalize=True,
)

In [ ]:
@torch.no_grad()
def predict_single_image(image, model, class_names, transform, device, topk=5):
    if isinstance(image, (str, Path)):
        with Image.open(image) as img:
            image = img.convert("RGB")
    elif isinstance(image, Image.Image):
        image = image.convert("RGB")
    else:
        raise TypeError("image must be a path or a PIL.Image")

    tensor = transform(image).unsqueeze(0).to(device)
    probs = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()
    order = np.argsort(probs)[::-1][:topk]
    return [(class_names[i], float(probs[i])) for i in order], image


sample_rows = test_df.sample(min(3, len(test_df)), random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(1, len(sample_rows), figsize=(5 * len(sample_rows), 4.8))
axes = np.atleast_1d(axes)

inference_rows = []
for ax, row in zip(axes, sample_rows.itertuples()):
    top_predictions, image = predict_single_image(
        DATA_ROOT / row.rel_path,
        best_model,
        CLASS_NAMES,
        eval_transform,
        DEVICE,
        topk=5,
    )
    top_name, top_prob = top_predictions[0]
    correct = top_name == row.class_name
    inference_rows.append({
        "rel_path": row.rel_path,
        "true_class": row.class_name,
        "pred_class": top_name,
        "confidence": top_prob,
        "correct": correct,
        "top5": ", ".join(f"{name} ({prob:.3f})" for name, prob in top_predictions),
    })

    top3_text = "\n".join(
        f"{rank}. {name} ({prob * 100:.1f}%)"
        for rank, (name, prob) in enumerate(top_predictions[:3], start=1)
    )
    ax.imshow(image)
    ax.axis("off")
    ax.set_title(
        f"True: {row.class_name}\n{top3_text}",
        fontsize=8,
        color="green" if correct else "darkred",
    )

fig.suptitle("MobileNetV3-Small inference on three test images")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mobilenetv3small_inference_examples.png", bbox_inches="tight")
plt.show()
plt.close(fig)

display(pd.DataFrame(inference_rows).round({"confidence": 4}))

## 10. Analisis error

In [ ]:
def build_prediction_frame(frame, probs, labels, preds):
    result = frame[["rel_path", "class_name", "source_group"]].copy()
    result["true_class"] = [CLASS_NAMES[int(i)] for i in labels]
    result["pred_class"] = [CLASS_NAMES[int(i)] for i in preds]
    result["confidence"] = probs.max(axis=1)
    result["correct"] = result["true_class"].eq(result["pred_class"])
    return result


def show_misclassified(predictions, model_name, output_stem, output_path, max_examples=6):
    errors = (
        predictions[~predictions["correct"]]
        .sort_values("confidence", ascending=False)
        .reset_index(drop=True)
    )
    print(f"{model_name}: {len(errors):,} salah dari {len(predictions):,} gambar test")
    if errors.empty:
        print("Tidak ada gambar yang salah diprediksi.")
        return errors

    display(errors[["rel_path", "true_class", "pred_class", "confidence"]].head(12))

    pair_counts = (
        errors.groupby(["true_class", "pred_class"])
        .size()
        .sort_values(ascending=False)
        .head(10)
        .rename("count")
        .reset_index()
    )
    print("Pasangan kelas yang paling sering tertukar:")
    display(pair_counts)

    examples = errors.head(max_examples)
    ncols = min(3, len(examples))
    nrows = math.ceil(len(examples) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.2 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (_, row) in zip(axes, examples.iterrows()):
        with Image.open(DATA_ROOT / row["rel_path"]) as img:
            ax.imshow(img.convert("RGB"))
        ax.set_title(
            f"True: {row['true_class']}\nPred: {row['pred_class']} ({row['confidence'] * 100:.1f}%)",
            fontsize=8,
            color="darkred",
        )
        ax.axis("off")
    for ax in axes[len(examples):]:
        ax.axis("off")
    fig.suptitle(f"{model_name} misclassified test examples")
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    errors.to_csv(RESULT_DIR / f"{output_stem}_misclassified.csv", index=False)
    return errors


predictions = build_prediction_frame(test_df, test_probs, test_labels, test_preds)
errors = show_misclassified(
    predictions,
    "MobileNetV3-Small",
    "mobilenetv3small",
    FIGURE_DIR / "mobilenetv3small_misclassified_examples.png",
)

## 11. Perbandingan dengan baseline

MobileNetV2 dan CNN dibaca dari `outputs/results/` (hasil eksperimen sebelumnya), MobileNetV3-Small
dari run ini.

In [ ]:
v3_comparison = pd.DataFrame([{
    "Model": "MobileNetV3-Small",
    "Test accuracy (%)": metrics["accuracy"] * 100,
    "Cross-entropy loss": metrics["loss"],
}]).set_index("Model")

v3_insights = pd.DataFrame([{
    "Model": "MobileNetV3-Small",
    "Top-3 accuracy (%)": metrics["top3_accuracy"] * 100,
    "3-class accuracy (%)": metrics["major_accuracy"] * 100,
    "3-class prob accuracy (%)": metrics["major_prob_accuracy"] * 100,
    "Macro F1": metrics["macro_f1"],
    "Parameters (M)": count_parameters(best_model) / 1e6,
}]).set_index("Model")

baseline_path = RESULT_DIR / "model_comparison.csv"
if baseline_path.exists():
    comparison = pd.concat(
        [v3_comparison, pd.read_csv(baseline_path).set_index("Model")]
    )
else:
    comparison = v3_comparison

insight_path = RESULT_DIR / "model_insights.csv"
if insight_path.exists():
    comparison_insights = pd.concat(
        [v3_insights, pd.read_csv(insight_path).set_index("Model")]
    )
else:
    comparison_insights = v3_insights

comparison.to_csv(RESULT_DIR / "mobilenetv3small_model_comparison.csv")
comparison_insights.to_csv(RESULT_DIR / "mobilenetv3small_model_insights.csv")

print("Metrik utama")
display(comparison.round(2))
print("Insight tambahan")
display(comparison_insights.round(3))

In [ ]:
colors = ["#54A24B", "#4C78A8", "#F58518"][:len(comparison)]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

comparison["Test accuracy (%)"].plot(kind="bar", ax=axes[0], color=colors)
axes[0].set_ylim(0, 100)
axes[0].set_ylabel("test accuracy (%)")
axes[0].set_title("Test accuracy")
axes[0].tick_params(axis="x", rotation=0)
for i, value in enumerate(comparison["Test accuracy (%)"]):
    axes[0].text(i, value + 1, f"{value:.2f}%", ha="center", fontsize=9)

comparison["Cross-entropy loss"].plot(kind="bar", ax=axes[1], color=colors)
axes[1].set_ylabel("cross-entropy loss")
axes[1].set_title("Test loss")
axes[1].tick_params(axis="x", rotation=0)
for i, value in enumerate(comparison["Cross-entropy loss"]):
    axes[1].text(i, value, f"{value:.3f}", ha="center", va="bottom", fontsize=9)

fig.suptitle("MobileNetV3-Small vs MobileNetV2 vs CNN")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mobilenetv3small_model_comparison.png", bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
history_files = {
    "MobileNetV3-Small": RESULT_DIR / "mobilenetv3small_history.csv",
    "MobileNetV2": RESULT_DIR / "mobilenetv2_history.csv",
    "CNN": RESULT_DIR / "cnn_baseline_history.csv",
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, path in history_files.items():
    if not path.exists():
        print(f"Skip {name}: {path.name} not found")
        continue
    frame = pd.read_csv(path)
    axes[0].plot(frame["epoch"], frame["val_loss"], label=name)
    axes[1].plot(frame["epoch"], frame["val_acc"], label=name)

axes[0].set_xlabel("epoch")
axes[0].set_ylabel("validation loss")
axes[0].set_title("Validation loss")
axes[0].legend()
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("validation accuracy")
axes[1].set_title("Validation accuracy")
axes[1].legend()

fig.suptitle("Validation curves: MobileNetV3-Small vs baselines")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mobilenetv3small_vs_baselines_curves.png", bbox_inches="tight")
plt.show()
plt.close(fig)

## 12. Simpan ringkasan

In [ ]:
summary = {
    "paper": {
        "title": "Application of MobileNetV2 to waste classification",
        "authors": "Yong, Ma, Sun, Du",
        "journal": "PLOS ONE 18(3): e0282336",
        "doi": "10.1371/journal.pone.0282336",
    },
    "model": {
        "arch": "mobilenet_v3_small",
        "parameters": count_parameters(best_model),
        "pretrained": PRETRAINED,
        "checkpoint": str(CHECKPOINT),
        "checkpoint_epoch": int(ckpt["epoch"]),
        "checkpoint_val_acc": float(ckpt["val_acc"]),
    },
    "dataset": {
        "root": str(DATA_ROOT),
        "split_manifest": str(SPLIT_MANIFEST),
        "num_classes": NUM_CLASSES,
        "num_images": int(len(df)),
        "split_sizes": {
            split_name: int((df["split"] == split_name).sum())
            for split_name in ("train", "val", "test")
        },
        "class_names": CLASS_NAMES,
        "major_names": MAJOR_NAMES,
        "class_to_major": CLASS_TO_MAJOR,
    },
    "training": {
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "momentum": MOMENTUM,
        "step_size": STEP_SIZE,
        "gamma": GAMMA,
        "test_batch_size": TEST_BATCH_SIZE,
        "seed": SEED,
        "device": str(DEVICE),
        "quick_test": QUICK_TEST,
    },
    "results": {
        "accuracy": metrics["accuracy"],
        "loss": metrics["loss"],
        "top3_accuracy": metrics["top3_accuracy"],
        "major_accuracy": metrics["major_accuracy"],
        "major_prob_accuracy": metrics["major_prob_accuracy"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
        "group_accuracy": metrics["group_accuracy"],
    },
    "artifacts": {
        "full_model": str(FULL_MODEL),
        "checkpoint": str(CHECKPOINT),
        "history": str(HISTORY_CSV),
        "metrics": str(RESULT_DIR / "mobilenetv3small_metrics.json"),
    },
    "versions": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "torchvision": __import__("torchvision").__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}
with open(RESULT_DIR / "mobilenetv3small_summary.json", "w") as handle:
    json.dump(jsonable(summary), handle, indent=2)
print("Wrote", RESULT_DIR / "mobilenetv3small_summary.json")

print("\nOutput files:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file() and path.stat().st_mtime >= time.time() - 3600:
        print(f"  {path.relative_to(OUTPUT_DIR)}")

## Catatan

- Arsitektur: MobileNetV3-Small, bukan MobileNetV2. Hyperparameter training, split, dan transform
  dibuat identik dengan eksperimen sebelumnya agar perbandingan tetap apples-to-apples.
- Dataset: 30 fine class, 3 major class (tanpa hazardous). Split 9.006 / 1.124 / 1.133 gambar unik.
- Artefak baru memakai prefix `mobilenetv3small_`; hasil MobileNetV2 dan CNN tidak ditimpa.